# COF and ABOD

Two variants for data where plain distances or densities struggle. The Connectivity-based Outlier Factor (COF) follows chains of neighbours, which suits elongated clusters. The Angle-Based Outlier Detector (ABOD) uses the angles to pairs of neighbours, which stay informative in higher dimensions.

In [ ]:
using OutlierDetection, OutlierDetectionNeighbors
using Plots, Statistics, Random

Random.seed!(3);

## COF on an Elongated Cluster

Points along a line, plus two points just off it. Their distance to the line is small, but they break the chain of neighbours.

In [ ]:
t = range(-5, 5; length = 200)
line = hcat(t, 0.5 .* t) .+ 0.1 .* randn(200, 2)
off  = [0.0 1.2; 2.0 -0.3]
X = vcat(line, off)
is_true = vcat(falses(200), trues(2))

_, lof_scores = OutlierDetection.fit(LOFDetector(k = 10), permutedims(X); verbosity = 0)
_, cof_scores = OutlierDetection.fit(COFDetector(k = 10), permutedims(X); verbosity = 0)

rank_of(s, i) = count(>=(s[i]), s)   # 1 = highest score
for i in findall(is_true)
    println("Point $i  LOF rank: ", rank_of(lof_scores, i), "   COF rank: ", rank_of(cof_scores, i))
end

Both detectors rank the two points at or near the top here. COF's advantage over LOF grows when the clusters are thin and sparsely sampled, where density estimates become unreliable.

In [ ]:
scatter(X[:, 1], X[:, 2]; zcolor = cof_scores, markerstrokewidth = 0,
    xlabel = "Feature 1", ylabel = "Feature 2", title = "COF scores", label = "")

## ABOD in Higher Dimensions

For a point inside the data, the angles to pairs of other points vary widely. For an outlier, all other points lie in roughly the same direction, so the angles vary little. ABOD scores are the **negated** variance of those angles, so outliers score close to 0 and normal points are more negative. Higher is still more anomalous.

In [ ]:
X10 = vcat(randn(300, 10), 3.5 .* sign.(randn(5, 10)))
is_true10 = vcat(falses(300), trues(5))

_, abod_scores = OutlierDetection.fit(ABODDetector(k = 10), permutedims(X10); verbosity = 0)
_, knn_scores  = OutlierDetection.fit(KNNDetector(k = 10), permutedims(X10); verbosity = 0)

top(s) = s .> quantile(s, 1 - 5 / length(s))
println("ABOD found: ", sum(top(abod_scores) .& is_true10), " of 5")
println("KNN found:  ", sum(top(knn_scores) .& is_true10), " of 5")

With 10 features KNN still finds these outliers too. Distances become less informative as the number of features grows further, and that is where angle-based scores help most.

In [ ]:
histogram(abod_scores; bins = 40, xlabel = "ABOD score (negated variance)", ylabel = "Count", label = "")

## Enhanced ABOD

`enhanced = true` weights the angles by distance as well, which often sharpens the separation.

In [ ]:
_, abod_enh = OutlierDetection.fit(ABODDetector(k = 10, enhanced = true), permutedims(X10); verbosity = 0)
println("Enhanced ABOD found: ", sum(top(abod_enh) .& is_true10), " of 5")

## Summary
- COF follows chains of neighbours and suits elongated or curved clusters.
- ABOD uses angle variance and stays informative as the number of features grows.
- ABOD scores are negated variances: outliers are near 0, and higher still means more anomalous.
- Compare against KNN and LOF before choosing.